# Advanced Results Processing

This notebook demonstrates advanced techniques for extracting and processing ospgrillage results using [xarray](http://xarray.pydata.org/en/stable/). It builds on the [Super-T Tutorial](super_t_analysis_workflow.ipynb).

## Setup

We first recreate the Super-T bridge model from the tutorial. See that notebook for detailed explanations.

In [ ]:
import ospgrillage as og
import numpy as np

# Units
kilo, milli, N, m = 1e3, 1e-3, 1, 1
mm = milli * m
m2, m3, m4 = m**2, m**3, m**4
kN = kilo * N
MPa = N / mm**2
GPa = kilo * MPa

In [ ]:
concrete = og.create_material(material="concrete", code="AS5100-2017", grade="65MPa")

longitudinal_section = og.create_section(A=1.025*m2, J=0.1878*m3, Iz=0.3694*m4, Iy=0.3634*m4, Az=0.4979*m2, Ay=0.309*m2)
edge_longitudinal_section = og.create_section(A=0.934*m2, J=0.1857*m3, Iz=0.3478*m4, Iy=0.213602*m4, Az=0.444795*m2, Ay=0.258704*m2)
transverse_section = og.create_section(A=0.504*m2, J=5.22303e-3*m3, Iy=0.32928*m4, Iz=1.3608e-3*m4, Ay=0.42*m2, Az=0.42*m2, unit_width=True)
end_transverse_section = og.create_section(A=0.504/2*m2, J=2.5e-3*m3, Iy=2.73e-2*m4, Iz=6.8e-4*m4, Ay=0.21*m2, Az=0.21*m2, unit_width=True)

longitudinal_beam = og.create_member(section=longitudinal_section, material=concrete)
edge_longitudinal_beam = og.create_member(section=edge_longitudinal_section, material=concrete)
transverse_slab = og.create_member(section=transverse_section, material=concrete)
end_transverse_slab = og.create_member(section=end_transverse_section, material=concrete)

In [ ]:
L = 33.5 * m
w = 11.565 * m

model = og.create_grillage(
    bridge_name="Super-T 33_5m", long_dim=L, width=w, skew=0,
    num_long_grid=7, num_trans_grid=11, edge_beam_dist=1.05 * m,
)
model.set_member(longitudinal_beam, member="interior_main_beam")
model.set_member(longitudinal_beam, member="exterior_main_beam_1")
model.set_member(longitudinal_beam, member="exterior_main_beam_2")
model.set_member(edge_longitudinal_beam, member="edge_beam")
model.set_member(transverse_slab, member="transverse_slab")
model.set_member(end_transverse_slab, member="start_edge")
model.set_member(end_transverse_slab, member="end_edge")
model.create_osp_model(pyfile=False)

In [ ]:
# Dead loads
beam_mag = 22.4 * kN / m
DL_self_weight = og.create_load_case(name="Super T self weight")
for z_pos in model.Mesh_obj.noz[1:-1]:
    p1 = og.create_load_vertex(x=0, z=z_pos, p=beam_mag)
    p2 = og.create_load_vertex(x=L, z=z_pos, p=beam_mag)
    DL_self_weight.add_load(og.create_load(loadtype="line", point1=p1, point2=p2, name="SW"))
model.add_load_case(DL_self_weight)

# Overlay slab
overlay = og.create_load(loadtype="patch", name="overlay",
    point1=og.create_load_vertex(x=0, z=0, p=4.32*kN/m2),
    point2=og.create_load_vertex(x=L, z=0, p=4.32*kN/m2),
    point3=og.create_load_vertex(x=L, z=w, p=4.32*kN/m2),
    point4=og.create_load_vertex(x=0, z=w, p=4.32*kN/m2))
DL_overlay = og.create_load_case(name="Overlay self weight")
DL_overlay.add_load(overlay)
model.add_load_case(DL_overlay)

# SIDL
asphalt = og.create_load(loadtype="patch", name="asphalt",
    point1=og.create_load_vertex(x=0, z=0, p=1.43*kN/m2),
    point2=og.create_load_vertex(x=L, z=0, p=1.43*kN/m2),
    point3=og.create_load_vertex(x=L, z=w, p=1.43*kN/m2),
    point4=og.create_load_vertex(x=0, z=w, p=1.43*kN/m2))
left_barrier = og.create_load(loadtype="line", name="left barrier",
    point1=og.create_load_vertex(x=0, z=0, p=6.54*kN/m),
    point2=og.create_load_vertex(x=L, z=0, p=6.54*kN/m))
right_barrier = og.create_load(loadtype="line", name="right barrier",
    point1=og.create_load_vertex(x=0, z=w, p=6.54*kN/m),
    point2=og.create_load_vertex(x=L, z=w, p=6.54*kN/m))
SIDL = og.create_load_case(name="SIDL")
SIDL.add_load(asphalt)
SIDL.add_load(left_barrier)
SIDL.add_load(right_barrier)
model.add_load_case(SIDL)

# M1600 traffic
gap = 6.25 * m
z_coord = [w/2, w/2 - 3.2*m, w/2 + 3.2*m]
alf = [1.0, 0.8, 0.4]
dla = 1.3
udl_width = 3.2 * m
udl_mag = 6*kN/m / udl_width

m1600_lc_list = [og.create_load_case(name=f"M1600 L{i+1}") for i in range(3)]
M1600_moving_loads = []
for i in range(3):
    vehicle = og.create_load_model(model_type="M1600", gap=gap).create()
    vehicle.set_global_coord(og.Point(0, 0, z_coord[i]))
    m1600_lc_list[i].add_load(vehicle, load_factor=alf[i])
    M1600_moving_loads.append(vehicle)
    udl = og.create_load(loadtype="patch", name="UDL",
        point1=og.create_load_vertex(x=-L, z=z_coord[i]-udl_width/2, p=udl_mag),
        point2=og.create_load_vertex(x=2*L, z=z_coord[i]-udl_width/2, p=udl_mag),
        point3=og.create_load_vertex(x=2*L, z=z_coord[i]+udl_width/2, p=udl_mag),
        point4=og.create_load_vertex(x=-L, z=z_coord[i]+udl_width/2, p=udl_mag))
    m1600_lc_list[i].add_load(udl, load_factor=alf[i])
    model.add_load_case(m1600_lc_list[i], load_factor=dla)

# Moving loads
path = og.create_moving_path(
    start_point=og.create_point(x=-25, y=0, z=0),
    end_point=og.Point(L, 0, 0))
for i in range(3):
    ml = og.create_moving_load(name=f"Moving M1600 L{i+1}")
    ml.set_path(path)
    ml.add_load(M1600_moving_loads[i])
    model.add_load_case(ml)

In [ ]:
model.analyze()

## Understanding the Results Dataset

The `get_results()` method returns an xarray `Dataset` containing multiple `DataArray` objects. Each DataArray is indexed along named dimensions.

In [ ]:
results = model.get_results()
results

The Dataset contains:
- `displacements` — indexed by `(Loadcase, Node, Component)` where Component is `dx, dy, dz, rx, ry, rz`
- `forces` — indexed by `(Loadcase, Element, Component)` where Component is `Mx_i, Mx_j, My_i, ...`
- `ele_nodes` — maps Element tags to their `i` and `j` nodes

The `Loadcase` dimension includes both static and moving load case names. Each moving load increment is stored as a separate load case named `"<name> at global position [x,y,z]"`, so even a modest model can have hundreds of load cases. Rather than printing them all, it's useful to summarise:

In [ ]:
# Summarise the load cases in the results
loadcases = list(results.coords["Loadcase"].values)
print(f"Total load cases: {len(loadcases)}")

# Separate static from moving (moving load names contain "at global position")
static_lc = [lc for lc in loadcases if "at global position" not in lc]
moving_lc = [lc for lc in loadcases if "at global position" in lc]

print(f"\nStatic load cases ({len(static_lc)}):")
for lc in static_lc:
    print(f"  - {lc}")

print(f"\nMoving load increments: {len(moving_lc)}")
if moving_lc:
    print(f"  First: {moving_lc[0]}")
    print(f"  Last:  {moving_lc[-1]}")

print(f"\nForce components: {list(results.coords['Component'].values)}")

In [ ]:
# Access a specific DataArray
displacements = results.displacements
forces = results.forces
print(f"Displacements shape: {displacements.shape}")
print(f"Forces shape: {forces.shape}")

## Selecting Results with `.sel()`

xarray's `.sel()` method is the primary way to extract specific results. You can select along any dimension.

In [ ]:
# Select vertical displacement for a specific load case
dy_sw = results.displacements.sel(Loadcase="Super T self weight", Component="dy")
print(f"Self-weight deflections (first 5 nodes): {dy_sw.values[:5]}")

In [ ]:
# Select bending moment for specific elements
member_name = "exterior_main_beam_1"
ext_beam_elements = model.get_element(member=member_name, options="elements")
ext_beam_nodes = model.get_element(member=member_name, options="nodes")

mz = results.forces.sel(Loadcase="M1600 L1", Component="Mz_i", Element=ext_beam_elements)
print(f"M1600 L1 bending moments on Beam 1 (kNm):\n{mz.values / 1000}")

In [ ]:
# Select multiple load cases at once
multi_lc = results.forces.sel(
    Loadcase=["M1600 L1", "M1600 L2", "M1600 L3"],
    Component="Mz_i",
    Element=ext_beam_elements,
)
print(f"Shape: {multi_lc.shape}  (3 load cases x {len(ext_beam_elements)} elements)")

## Filtering by Load Case Type

Use the `load_case` keyword to retrieve only specific load cases. This is useful when you want to work with static or moving results separately.

In [ ]:
# Static load cases only
static_results = model.get_results(
    load_case=["Super T self weight", "SIDL", "Overlay self weight",
               "M1600 L1", "M1600 L2", "M1600 L3"]
)
static_results

In [ ]:
# Moving load results for Lane 2
moving_results = model.get_results(load_case="Moving M1600 L2")
print(f"Moving load has {len(moving_results.coords['Loadcase'])} incremental positions")
moving_results

## Load Combinations

Pass a dictionary of `{load_case_name: factor}` to compute factored combinations on the fly.

In [ ]:
combination = {
    "Super T self weight": 1.0,
    "M1600 L1": 1.0,
    "M1600 L2": 1.0,
    "M1600 L3": 1.0,
    "SIDL": 1.3,
    "Overlay self weight": 1.0,
}
comb_results = model.get_results(combinations=combination)
comb_results

In [ ]:
# Extract maximum combined bending moment
comb_mz = comb_results.forces.sel(Component="Mz_i", Element=ext_beam_elements)
print(f"Max combined BM on Beam 1 = {max(comb_mz.values / 1000):.2f} kN m")

## Envelopes

The `create_envelope()` function finds the maximum (or minimum) response across all load cases for each element. This is essential for design — it tells you the worst-case response regardless of which load case caused it.

In [ ]:
# Envelope of static load cases
envelope = og.create_envelope(ds=static_results, load_effect="Mz_i", array="forces")
bending_env = envelope.get()
bending_env

In [ ]:
# Envelope of moving load positions
moving_envelope = og.create_envelope(ds=moving_results, load_effect="Mz_i", array="forces")
moving_env = moving_envelope.get()
moving_env.sel(Element=ext_beam_elements, Component="Mz_i")

### Query Mode — Which Load Case Governs?

Set `query_mode=True` to find out which load case produced the maximum at each element.

In [ ]:
query_envelope = og.create_envelope(
    ds=moving_results, load_effect="Mz_i", array="forces", query_mode=True
)
query_env = query_envelope.get()

# Which moving load position gives max BM on each element of Beam 1?
governing = query_env.sel(Element=ext_beam_elements, Component="Mz_i")
governing

## Custom Post-Processing

Extract numpy arrays for custom calculations.

In [ ]:
# Sum node forces to compute support reactions
dy_all = results.displacements.sel(Loadcase="Super T self weight", Component="dy")
support_nodes = model.get_nodes(options="support_nodes")
reactions = dy_all.sel(Node=support_nodes) if support_nodes else dy_all
print(f"Support node deflections: {reactions.values}")

In [ ]:
# Compare maximum bending across all beams
for member in ["exterior_main_beam_1", "interior_main_beam", "exterior_main_beam_2"]:
    elements = model.get_element(member=member, options="elements")
    mz = results.forces.sel(Loadcase="M1600 L1", Component="Mz_i", Element=elements)
    print(f"{member}: max Mz = {max(mz.values) / 1000:.1f} kN m")

## Summary

Key techniques covered:

| Technique | Method |
|---|---|
| Select specific results | `results.forces.sel(Loadcase=..., Component=..., Element=...)` |
| Filter load cases | `model.get_results(load_case=[...])` |
| Load combinations | `model.get_results(combinations={...})` |
| Envelopes | `og.create_envelope(ds=..., load_effect=..., array=...)` |
| Query governing case | `og.create_envelope(..., query_mode=True)` |
| Extract numpy arrays | `.values` on any DataArray |